In [1]:
# Cell 1: 安装依赖
!pip install librosa tqdm
print("✅ 依赖库安装完成！")

Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 26.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 21.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.9/208.9 kB 36.4 MB/s eta 0:00:00
✅ 依赖库安装完成！


In [2]:
# Cell 2: 数据搬运 (保持不变)
import moxing as mox
import os
import time

OBS_PATH = "obs://bullying2/raw_data"
LOCAL_PATH = "/cache/dataset"

if not os.path.exists(LOCAL_PATH):
    os.makedirs(LOCAL_PATH)

print(f"🚀 正在从 {OBS_PATH} 下载数据...")
start = time.time()

try:
    mox.file.copy_parallel(OBS_PATH, LOCAL_PATH)
    print(f"✅ 下载完成！耗时: {time.time() - start:.2f} 秒")
    print("📂 本地目录检查:", os.listdir(LOCAL_PATH))
except Exception as e:
    print(f"❌ 下载失败: {e}")

INFO:root:Using MoXing-v2.3.7-eadf950b
INFO:root:Using OBS-Python-SDK-3.25.3
INFO:root:Using OBS-C-SDK-2.23.1


🚀 正在从 obs://bullying2/raw_data 下载数据...


INFO:root:Multiprocessing connection patch for bpo-17560 not applied, not an applicable Python version: 3.11.10 (main, Oct  3 2024, 07:21:51) [GCC 11.2.0]
INFO:root:Listing OBS: 1000
INFO:root:Listing OBS: 2000
INFO:root:Listing OBS: 3000
INFO:root:Listing OBS: 4000
INFO:root:Listing OBS: 5000
INFO:root:Listing OBS: 6000
INFO:root:Listing OBS: 7000
INFO:root:Listing OBS: 8000
INFO:root:Listing OBS: 9000
INFO:root:Listing OBS: 10000
INFO:root:Listing OBS: 11000
INFO:root:Listing OBS: 12000
INFO:root:Listing OBS: 13000
INFO:root:Listing OBS: 14000
INFO:root:Listing OBS: 15000
INFO:root:Listing OBS: 16000
INFO:root:Listing OBS: 17000
INFO:root:Listing OBS: 18000
INFO:root:Listing OBS: 19000
INFO:root:Listing OBS: 20000
INFO:root:Listing OBS: 21000
INFO:root:Listing OBS: 22000
INFO:root:Listing OBS: 23000
INFO:root:Listing OBS: 24000
INFO:root:Listing OBS: 25000
INFO:root:Listing OBS: 26000
INFO:root:Listing OBS: 27000
INFO:root:Listing OBS: 28000
INFO:root:Listing OBS: 29000
INFO:root:Lis

✅ 下载完成！耗时: 128.55 秒
📂 本地目录检查: ['S01', 'MIVIA_DB4_dist']


In [ ]:
import os
import glob
import numpy as np
import zipfile

# ========== 配置 ==========
ZIP_DIR    = '../data/samples/MMFi'   # S01-S03 zip 所在目录
EXTRACT_DIR = '../data/samples/MMFi/extracted'
OUTPUT_DIR  = '../data'
os.makedirs(EXTRACT_DIR, exist_ok=True)

# 解压 zip
for zf in sorted(glob.glob(os.path.join(ZIP_DIR, 'S*.zip'))):
    with zipfile.ZipFile(zf) as z:
        z.extractall(EXTRACT_DIR)
    print(f'解压: {os.path.basename(zf)}')

# 动作标签
ACTION_LABELS = {
    'A02': 1, 'A03': 1, 'A04': 1,   # 霸凌动作
    'A09': 0, 'A10': 0, 'A11': 0,   # 正常动作
}

def read_bin(path):
    try:
        data = np.fromfile(path, dtype=np.float32)
        if len(data) % 5 != 0:
            return None
        pts = data.reshape(-1, 5)  # (N, 5): x,y,z,doppler,SNR
        if len(pts) == 0:
            return None
        # 截断或补零到64点
        if len(pts) >= 64:
            pts = pts[:64]
        else:
            pad = np.zeros((64 - len(pts), 5), dtype=np.float32)
            pts = np.concatenate([pts, pad], axis=0)
        return pts.T  # (5, 64)
    except Exception:
        return None

# 找到所有受试者目录 S01-S03
subject_dirs = sorted(glob.glob(os.path.join(EXTRACT_DIR, 'S*')))
print(f'共找到 {len(subject_dirs)} 个受试者目录')

radar_X, radar_y = [], []
for act, lbl in ACTION_LABELS.items():
    files = []
    for s_dir in subject_dirs:
        files += glob.glob(os.path.join(s_dir, '**', act, 'mmwave', '*.bin'), recursive=True)
    print(f'   -> {act}: {len(files)} 帧')
    for f in files:
        pc = read_bin(f)
        if pc is None:
            continue
        radar_X.append(pc)
        radar_y.append(lbl)

if radar_X:
    np.save(os.path.join(OUTPUT_DIR, 'radar_X.npy'), np.stack(radar_X).astype(np.float32))
    np.save(os.path.join(OUTPUT_DIR, 'radar_y.npy'), np.array(radar_y, dtype=np.int32))
    labels = np.array(radar_y)
    print(f'雷达: {len(radar_X)} 个样本，维度: 5 (x, y, z, doppler, SNR)')
    print(f'   正常(0): {(labels==0).sum()}, 霸凌(1): {(labels==1).sum()}')
    print(f'数据已保存至: {OUTPUT_DIR}')
else:
    print('未找到数据，请检查 ZIP_DIR 路径和文件结构')


In [4]:
# Cell 4: 高级模型架构 - PointNet++雷达分支 + 跨模态注意力
import mindspore
import mindspore.nn as nn
import mindspore.ops as ops
from mindspore import context, Tensor

context.set_context(mode=context.GRAPH_MODE, device_target="Ascend")

# ========== PointNet++ 核心模块 ==========
class PointNetSetAbstraction(nn.Cell):
    """PointNet++ Set Abstraction层 - 简化版适配MindSpore 2.4.10"""
    def __init__(self, in_channel, mlp):
        super().__init__()
        self.mlp_convs = nn.CellList()
        self.mlp_bns = nn.CellList()
        
        last_channel = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv1d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm1d(out_channel))
            last_channel = out_channel
        
        self.relu = nn.ReLU()
    
    def construct(self, x):
        for i, conv in enumerate(self.mlp_convs):
            bn = self.mlp_bns[i]
            x = self.relu(bn(conv(x)))
        return x

# ========== 雷达分支：PointNet++ ==========
class RadarPointNetBranch(nn.Cell):
    """雷达点云处理分支"""
    def __init__(self):
        super().__init__()
        self.sa1 = PointNetSetAbstraction(5, [32, 64])  # 🔥 改为5维输入 (x, y, z, doppler, SNR)
        self.sa2 = PointNetSetAbstraction(64, [64, 128])
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
    
    def construct(self, x):
        x = self.sa1(x)
        x = self.sa2(x)
        x = self.global_pool(x)
        x = self.flatten(x)
        return x

# ========== 音频分支 ==========
class AudioCNNBranch(nn.Cell):
    def __init__(self):
        super().__init__()
        self.conv = nn.SequentialCell([
            nn.Conv1d(1, 32, 64, stride=16, pad_mode='valid'),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(32, 64, 3, pad_mode='same'),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(64, 128, 3, pad_mode='same'),
            nn.ReLU(), nn.AdaptiveAvgPool1d(1), nn.Flatten()
        ])
    
    def construct(self, x):
        return self.conv(x)

# ========== 跨模态注意力 ==========
class CrossModalAttention(nn.Cell):
    def __init__(self, dim=128):
        super().__init__()
        self.fc_radar = nn.Dense(dim, dim)
        self.fc_audio = nn.Dense(dim, dim)
        self.fc_out = nn.Dense(dim, dim)
        self.sigmoid = nn.Sigmoid()
    
    def construct(self, radar_feat, audio_feat):
        attn = self.sigmoid(self.fc_radar(radar_feat) + self.fc_audio(audio_feat))
        out = self.fc_out(attn * (radar_feat + audio_feat))
        return out

# ========== 完整模型 ==========
class AscendSentinel2(nn.Cell):
    def __init__(self):
        super().__init__()
        self.radar_branch = RadarPointNetBranch()
        self.audio_branch = AudioCNNBranch()
        self.cross_attention = CrossModalAttention(128)
        self.classifier = nn.SequentialCell([
            nn.Dense(384, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.3), nn.Dense(256, 128), nn.ReLU(),
            nn.Dense(128, 2)
        ])
    
    def construct(self, radar, audio):
        radar_feat = self.radar_branch(radar)
        audio_feat = self.audio_branch(audio)
        cross_feat = self.cross_attention(radar_feat, audio_feat)
        fused = ops.concat([radar_feat, audio_feat, cross_feat], axis=1)
        return self.classifier(fused)

print("✅ 高级模型架构定义完成！")
print("📊 雷达输入维度: 5 (x, y, z, doppler, SNR)")

[WARNING] ME(29518:281473636413664,MainProcess):2026-03-19-21:44:16.300.0 [mindspore/run_check/_check_version.py:409] Can not find the tbe operator implementation(need by mindspore-ascend). Please check whether the Environment Variable PYTHONPATH is set. For details, refer to the installation guidelines: https://www.mindspore.cn/install
[WARNING] ME(29518:281473636413664,MainProcess):2026-03-19-21:44:26.744.000 [mindspore/run_check/_check_version.py:409] Can not find the tbe operator implementation(need by mindspore-ascend). Please check whether the Environment Variable PYTHONPATH is set. For details, refer to the installation guidelines: https://www.mindspore.cn/install
[WARNING] ME(29518:281473636413664,MainProcess):2026-03-19-21:44:26.745.000 [mindspore/context.py:1418] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.
[WARNING] ME(29518:281473636413664,MainProcess):2026-03-

✅ 高级模型架构定义完成！
📊 雷达输入维度: 5 (x, y, z, doppler, SNR)


In [5]:
# Cell 5: 数据集与训练 (添加训练/验证集划分)
import os
import numpy as np
from mindspore.dataset import GeneratorDataset
from mindspore.train.callback import LossMonitor, TimeMonitor

class MultiModalDataset:
    def __init__(self, data_dir, split='train', train_ratio=0.8):
        radar_X = np.load(os.path.join(data_dir, "radar_X.npy"))
        radar_y = np.load(os.path.join(data_dir, "radar_y.npy"))
        self.audio_X = np.load(os.path.join(data_dir, "audio_X.npy"))
        
        # 划分训练集和验证集
        n_samples = len(radar_X)
        indices = np.random.permutation(n_samples)
        split_idx = int(n_samples * train_ratio)
        
        if split == 'train':
            self.indices = indices[:split_idx]
        else:  # val
            self.indices = indices[split_idx:]
        
        self.radar_X = radar_X[self.indices]
        self.radar_y = radar_y[self.indices]
        self.pos_indices = np.where(self.radar_y == 1)[0]
        
        print(f"📊 {split.upper()}集: {len(self.indices)}个样本 (正样本:{len(self.pos_indices)}, 负样本:{len(self.indices)-len(self.pos_indices)})")
    
    def __getitem__(self, index):
        radar_sample = self.radar_X[index]
        label = self.radar_y[index]
        radar_tensor = radar_sample.transpose(1, 0).astype(np.float32)
        
        if label == 1 and len(self.audio_X) > 0:
            rand_idx = np.random.randint(0, len(self.audio_X))
            audio_sample = self.audio_X[rand_idx]
        else:
            audio_sample = np.random.randn(16000) * 0.01
        
        audio_tensor = audio_sample.reshape(1, 16000).astype(np.float32)
        return radar_tensor, audio_tensor, np.array(label, dtype=np.int32)
    
    def __len__(self):
        return len(self.radar_X)

class TrainOneStepCell(nn.Cell):
    def __init__(self, network, optimizer, loss_fn):
        super().__init__(auto_prefix=False)
        self.network = network
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.grad_fn = mindspore.value_and_grad(self.forward_fn, None, self.optimizer.parameters)
    
    def forward_fn(self, radar, audio, label):
        logits = self.network(radar, audio)
        return self.loss_fn(logits, label)
    
    def construct(self, radar, audio, label):
        loss, grads = self.grad_fn(radar, audio, label)
        self.optimizer(grads)
        return loss

data_dir = "/home/ma-user/work/Ascend_Processed_Data"

# 创建训练集和验证集
print("=" * 70)
print("📦 数据集划分")
print("=" * 70)
train_dataset = MultiModalDataset(data_dir, split='train', train_ratio=0.8)
val_dataset = MultiModalDataset(data_dir, split='val', train_ratio=0.8)

train_ds = GeneratorDataset(train_dataset, ["radar", "audio", "label"], shuffle=True)
train_ds = train_ds.batch(32, drop_remainder=True)

net = AscendSentinel2()
loss_fn = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
optimizer = nn.Adam(net.trainable_params(), learning_rate=0.001, weight_decay=1e-4)

train_net = TrainOneStepCell(net, optimizer, loss_fn)
train_net.set_train()

from mindspore import Model
model = Model(train_net)

print("\n🚀 开始训练昇腾哨兵2.0...")
print("=" * 70)
model.train(epoch=15, train_dataset=train_ds, callbacks=[LossMonitor(20), TimeMonitor()], dataset_sink_mode=False)
print("=" * 70)
print("✅ 训练完成！")

# 保存.ckpt文件
ckpt_path = "/home/ma-user/work/ascend_sentinel_final.ckpt"
mindspore.save_checkpoint(net, ckpt_path)
print(f"💾 模型已保存: {ckpt_path}")

📦 数据集划分
📊 TRAIN集: 1900个样本 (正样本:949, 负样本:951)
📊 VAL集: 476个样本 (正样本:234, 负样本:242)


[WARNING] ME(29518:281473636413664,MainProcess):2026-03-19-21:44:39.904.000 [mindspore/nn/layer/basic.py:174] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.



🚀 开始训练昇腾哨兵2.0...


[WARNING] ME(29518:281473636413664,MainProcess):2026-03-19-21:44:40.311.000 [mindspore/nn/layer/basic.py:200] For Dropout, this parameter `keep_prob` will be deprecated, please use `p` instead.
[WARNING] CORE(29518,ffffb01cb0e0,python):2026-03-19-21:44:40.316.960 [mindspore/core/utils/ms_context.cc:537] GetJitLevel] Set jit level to O2 for rank table startup method.
[ERROR] CORE(29518,ffffb01cb0e0,python):2026-03-19-21:44:42.409.857 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_29518/3142988853.py]
[WARNING] CORE(29518,ffffb01cb0e0,python):2026-03-19-21:44:42.409.910 [mindspore/core/utils/info.cc:94] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_29518/3142988853.py' may not exists.


......epoch: 1 step: 20, loss is 0.018279176205396652
epoch: 1 step: 40, loss is 0.0015079181175678968
Train epoch time: 101813.376 ms, per step time: 1725.650 ms
epoch: 2 step: 1, loss is 0.0064748432487249374
epoch: 2 step: 21, loss is 0.0013773568207398057
epoch: 2 step: 41, loss is 0.0002859502856153995
Train epoch time: 17328.501 ms, per step time: 293.703 ms
epoch: 3 step: 2, loss is 0.0002363913954468444
epoch: 3 step: 22, loss is 4.750712469103746e-05
epoch: 3 step: 42, loss is 0.00010313135135220364
Train epoch time: 17331.716 ms, per step time: 293.758 ms
epoch: 4 step: 3, loss is 8.606084156781435e-05
epoch: 4 step: 23, loss is 5.054267239756882e-05
epoch: 4 step: 43, loss is 0.00012886816693935543
Train epoch time: 17322.522 ms, per step time: 293.602 ms
epoch: 5 step: 4, loss is 0.00012649594282265753
epoch: 5 step: 24, loss is 7.477916369680315e-05
epoch: 5 step: 44, loss is 0.0051559205166995525
Train epoch time: 17332.602 ms, per step time: 293.773 ms
epoch: 6 step: 5, 

In [ ]:
# Cell 6: 正反例验证（各10个）
print("=" * 70)
print("【验证集正反例分析】")
print("=" * 70)

# 正样本验证
pos_indices = np.where(dataset_val.radar_y == 1)[0]
num_pos_test = min(10, len(pos_indices))
print(f"\n【正样本验证】异常行为检测 (测试{num_pos_test}个，验证集共{len(pos_indices)}个)")
print("-" * 70)

pos_correct = 0
pos_samples = np.random.choice(pos_indices, num_pos_test, replace=False) if len(pos_indices) >= num_pos_test else pos_indices

for i, idx in enumerate(pos_samples):
    pred, probs, true_label, _, _ = predict_sample(idx)
    status = "✅" if pred == true_label else "❌"
    pos_correct += (pred == true_label)
    print(f"样本{idx:4d} | 真值:异常 -> 预测:{'异常' if pred==1 else '正常'} | 置信度:{probs[1]:.3f} | {status}")

print(f"\n✅ 正样本准确率: {pos_correct}/{len(pos_samples)} = {pos_correct/len(pos_samples)*100:.2f}%")

# 负样本验证
neg_indices = np.where(dataset_val.radar_y == 0)[0]
num_neg_test = min(10, len(neg_indices))
print(f"\n【负样本验证】正常行为检测 (测试{num_neg_test}个，验证集共{len(neg_indices)}个)")
print("-" * 70)

neg_correct = 0
neg_samples = np.random.choice(neg_indices, num_neg_test, replace=False) if len(neg_indices) >= num_neg_test else neg_indices

for i, idx in enumerate(neg_samples):
    pred, probs, true_label, _, _ = predict_sample(idx)
    status = "✅" if pred == true_label else "❌"
    neg_correct += (pred == true_label)
    print(f"样本{idx:4d} | 真值:正常 -> 预测:{'正常' if pred==0 else '异常'} | 置信度:{probs[0]:.3f} | {status}")

print(f"\n✅ 负样本准确率: {neg_correct}/{len(neg_samples)} = {neg_correct/len(neg_samples)*100:.2f}%")

print("\n" + "=" * 70)
print(f"【验证集性能】")
total = len(pos_samples) + len(neg_samples)
correct = pos_correct + neg_correct
print(f"  总样本数: {total}")
print(f"  正确预测: {correct}")
print(f"  总体准确率: {correct/total*100:.2f}%")
print(f"  召回率(异常): {pos_correct/len(pos_samples)*100:.2f}%")
print(f"  特异度(正常): {neg_correct/len(neg_samples)*100:.2f}%")
print("=" * 70)
